In [45]:
import numpy as np
import polars as pl


import matplotlib.pyplot as plt
import matplotlib.cm as cm
import time

import pandas as pd
from extinction import fitzpatrick99


from scipy.stats import kurtosis, skew, spearmanr, linregress

from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern, ConstantKernel

from sklearn.feature_selection import VarianceThreshold
from sklearn.metrics import classification_report, f1_score

from romae_mallorn.romae_contrastive import RoMAEPreTrainingContrastive, RandomMasking
from romae_mallorn.dataset import MallornDatasetwLabelTrimMask
from romae_mallorn.env_config import MallornConfigContrastiveEnv
from romae_mallorn.utils import override_encoder_size, load_from_checkpoint_contrastive


from romae.model import RoMAEForPreTrainingConfig,EncoderConfig

import torch.nn as nn
import torch

https://github.com/maymeridian/Tidal-Disruption-Classification/blob/competition_winner/src/data_loader.py#L76

## Process:
    1. GET DATA
    
        X_train, y_train = get_prepared_dataset('train')

    2. RUN TRAINING (Using 5-Fold CV + Class Weights + Auto-Tuning)
    
        model, score, threshold = train_with_cv(model_name, X_train, y_train)


 get_prepared_Dataset:
 
     - load_lightcurves (.csv files one row = one obs (with source_id col) I think; should be all combined)
     
     - extract_features:
     
       x log_df = get_log_data(dataset_type) (get pandas DataFrame from csv)
       
       x lc_df = apply_deextinction(lc_df, log_df)
       
       * lc_clean = apply_quality_cuts(lc_df)
       
       * Fit 2D GP and get GP features: get_gp_features
       
       * if Z: additional set of features


   
df vs log_df? (log_df has target / labels from MALLORN I think)

In [2]:
# Effective wavelengths (Angstroms) for LSST filters
FILTER_WAVELENGTHS = {
    'u': 3641,
    'g': 4704,
    'r': 6155,
    'i': 7504,
    'z': 8695,
    'y': 10056,    
    'Y': 10056
}

def apply_deextinction_parq(df):
    if 'Flux_Corrected' in df.columns: 
        return df
    if 'EBV' not in df.columns and 'MWEBV' not in df.columns:    
        df = df.with_columns([
            pl.col('FLUXCAL').alias('Flux_Corrected'),
            pl.col('FLUXCALERR').alias('Flux_err_Corrected')
        ])
        
        return df
    # ebv_map = log_df.set_index('object_id')['EBV']
    # df['EBV'] = df['object_id'].map(ebv_map)

    ebv_key = 'EBV' if 'EBV' in df.columns else 'MWEBV'
    
    unique_filters = list(FILTER_WAVELENGTHS.keys())
    unique_wls = np.array([FILTER_WAVELENGTHS[f] for f in unique_filters], dtype=float)
    ext_factors = fitzpatrick99(unique_wls, 1.0)
    ext_map = dict(zip(unique_filters, ext_factors))

    def correct_flux(fluxes, bands, ebv):
        a_lambda = np.array([ext_map[b] for b in bands]) * (ebv * 3.1)
        correction = 10 ** (a_lambda / 2.5)
        return (np.array(fluxes) * correction).tolist()

    df = df.with_columns([
        pl.struct(['FLUXCAL', 'BAND', ebv_key]).map_elements(
            lambda r: correct_flux(r['FLUXCAL'], r['BAND'], r[ebv_key]),
            return_dtype=pl.List(pl.Float64)
        ).alias('Flux_Corrected'),

        pl.struct(['FLUXCALERR', 'BAND', ebv_key]).map_elements(
            lambda r: correct_flux(r['FLUXCALERR'], r['BAND'], r[ebv_key]),
            return_dtype=pl.List(pl.Float64)
        ).alias('Flux_err_Corrected'),
    ])
    
    return df


In [3]:
def apply_quality_cuts(lc_df):
    # if 'SNR' not in lc_df.columns:
    #     safe_err = lc_df['Flux_err'].replace(0, 1e-5)
    #     lc_df['SNR'] = lc_df['Flux'] / safe_err
    ## I don't think they use this anywhere?

    # valid_mask = (lc_df['Flux'] > 0) 
    # counts = lc_df[valid_mask].groupby('object_id').size()
    # keep_ids = counts[counts >= 2].index
    # return lc_df[lc_df['object_id'].isin(keep_ids)].copy()

    ## This is basically removing/ignoring light-curves that have less than 2 positive FLUXCAL
    return lc_df.filter(pl.col('FLUXCAL').list.eval(pl.element() > 0).list.sum() >= 2)

In [4]:
def fit_2d_gp(obj_df):
    ## Assume a one row pl.DF for one obj 
    
    if 'Flux_Corrected' in obj_df.columns:
        y = obj_df['Flux_Corrected'][0].to_numpy()
        y_err = obj_df['Flux_err_Corrected'][0].to_numpy()
    else:
        y = obj_df['FLUXCAL'][0].to_numpy()
        y_err = obj_df['FLUXCALERR'][0].to_numpy()

    X = np.column_stack([obj_df['MJD'][0].to_numpy(),
                         np.array([FILTER_WAVELENGTHS[b] for b in obj_df['BAND'][0]])])

    y_scale = np.max(np.abs(y)) if np.max(np.abs(y)) > 0 else 1.0
    y_norm = y / y_scale
    y_err_norm = y_err / y_scale

    kernel = ConstantKernel(1.0) * Matern(length_scale=[100, 6000], nu=1.5)
    gp = GaussianProcessRegressor(kernel=kernel, alpha=y_err_norm**2, n_restarts_optimizer=0, random_state=42)
    gp.fit(X, y_norm)
    return gp, y_scale


In [5]:


def calculate_template_matching(t_grid, y_pred_g, peak_idx, peak_time, peak_flux):
    """Normalized TDE Shape Matching ONLY."""
    post_peak_indices = np.where(t_grid > peak_time)[0]
    match_tde = 10.0 
    
    if len(post_peak_indices) > 5 and peak_flux > 0:
        y_fade = y_pred_g[post_peak_indices]
        t_fade = t_grid[post_peak_indices] - peak_time
        
        y_norm = y_fade / peak_flux
        half_idx = np.argmax(y_norm < 0.5)
        
        if half_idx > 0:
            t_half = t_fade[half_idx]
            if t_half > 0.1: 
                t_norm = t_fade / t_half 
                y_temp_tde = 1.0 / (1.0 + (t_norm * (2**(1/1.67) - 1)))**1.67
                mask = t_norm < 3.0
                if mask.sum() > 2:
                    match_tde = np.sqrt(np.mean((y_norm[mask] - y_temp_tde[mask])**2))
                    
    return match_tde

def calculate_physics_wars(t_grid, y_pred_g, peak_idx, peak_time, peak_flux):
    # 1. TDE FIT
    post_peak_indices = np.where(t_grid > peak_time)[0]
    tde_error = 10.0 
    linear_decay_slope = 0.0 
    
    if len(post_peak_indices) > 5 and peak_flux > 0:
        y_real_fade = y_pred_g[post_peak_indices]
        t_fade = t_grid[post_peak_indices]
        dt = (t_fade - peak_time) + 10 
        
        y_ideal_tde = peak_flux * (dt / dt[0])**(-1.67)
        residuals_tde = (y_real_fade - y_ideal_tde) / peak_flux
        tde_error = np.mean(residuals_tde**2)

        try:
            log_t = np.log(dt)
            log_y = np.log(y_real_fade + 1e-9)
            # slope, _, _, _, _ = linregress(log_t, log_y)
            # linear_decay_slope = slope
            linear_decay_slope = slope if not np.isnan(slope) else 0.0
        except Exception:
            linear_decay_slope = 0.0

    # 2. RISE PHYSICS
    pre_peak_indices = np.where(t_grid < peak_time)[0]
    rise_fireball_error = 10.0
    pre_peak_var = 0.0 
    
    if len(pre_peak_indices) > 5 and peak_flux > 0:
        y_real_rise = y_pred_g[pre_peak_indices]
        t_rise = t_grid[pre_peak_indices]
        if len(t_rise) > 3:
            try:
                coeffs = np.polyfit(t_rise, y_real_rise, 2)
                p = np.poly1d(coeffs)
                residuals_rise = (y_real_rise - p(t_rise)) / peak_flux
                rise_fireball_error = np.mean(residuals_rise**2)
                pre_peak_var = np.var(residuals_rise)
            except Exception:
                pass

    fade_correlation = 0.0
    # if len(post_peak_indices) > 2:
    #     fade_correlation = np.corrcoef(t_grid[post_peak_indices], y_pred_g[post_peak_indices])[0, 1]
    if len(post_peak_indices) > 2:
        t_post = t_grid[post_peak_indices]
        y_post = y_pred_g[post_peak_indices]
        if np.std(y_post) > 0 and np.std(t_post) > 0:
            fade_correlation = np.corrcoef(t_post, y_post)[0, 1]
        else:
            fade_correlation = 0.0
            
    half_max = peak_flux / 2.0
    rise_idx_candidates = np.where((y_pred_g[:peak_idx] <= half_max))[0]
    t_half_rise = t_grid[rise_idx_candidates[-1]] if len(rise_idx_candidates) > 0 else t_grid[0]
    fade_idx_candidates = np.where((y_pred_g[peak_idx:] <= half_max))[0]
    t_half_fade = t_grid[peak_idx + fade_idx_candidates[0]] if len(fade_idx_candidates) > 0 else t_grid[-1]
    fwhm = t_half_fade - t_half_rise
    
    return tde_error, linear_decay_slope, rise_fireball_error, fade_correlation, fwhm, pre_peak_var


def get_gp_features(obj_id, obj_df):
    try:
        gp, y_scale = fit_2d_gp(obj_df)
    except Exception:
        return None

    params = gp.kernel_.get_params()
    try:
        ls_time = params.get('k2__length_scale', [0,0])[0]
        ls_wave = params.get('k2__length_scale', [0,0])[1]
        amplitude = np.sqrt(params.get('k1__constant_value', 0)) * y_scale
    except Exception:
        ls_time, ls_wave, amplitude = 0, 0, 0

    if 'Flux_Corrected' in obj_df.columns:
        flux_data = obj_df['Flux_Corrected'][0].to_numpy()
        flux_err = obj_df['Flux_err_Corrected'][0].to_numpy()
    else:
        flux_data = obj_df['FLUXCAL'][0].to_numpy()
        flux_err = obj_df['FLUXCALERR'][0].to_numpy()

    significant_negative = (flux_data < -3 * flux_err)
    negative_flux_fraction = significant_negative.sum() / len(flux_data) if len(flux_data) > 0 else 0.0

    significant_mask = flux_data > (3 * flux_err) 

    mjd_obj = obj_df['MJD'][0].to_numpy()
    band_obj = obj_df['BAND'][0].to_numpy()
    valband_obj = np.array([FILTER_WAVELENGTHS[b] for b in band_obj])
    
    detection_times = mjd_obj[significant_mask]
    
    total_survey_span = obj_df['MJD'][0].max() - obj_df['MJD'][0].min()
    
    if len(detection_times) > 4:
        t_10 = np.percentile(detection_times, 10)
        t_90 = np.percentile(detection_times, 90)
        robust_duration = t_90 - t_10
        duty_cycle = robust_duration / total_survey_span if total_survey_span > 0 else 0
    else:
        robust_duration = 0.0
        duty_cycle = 0.0

    flux_kurtosis = kurtosis(flux_data, fisher=True)
    flux_skew = skew(flux_data) 

    X_obs = np.column_stack([mjd_obj, valband_obj])
    y_gp_pred = gp.predict(X_obs) * y_scale
    safe_err = np.where(flux_err <= 0, 1e-5, flux_err)
    chi_sq_terms = ((flux_data - y_gp_pred) / safe_err) ** 2
    reduced_chi_square = np.mean(chi_sq_terms)
    reduced_chi_square = min(reduced_chi_square, 1000.0)

    # 100 Points Grid
    t_min, t_max = mjd_obj.min(), mjd_obj.max()
    t_grid = np.linspace(t_min, t_max, 100)
    g_wave = FILTER_WAVELENGTHS['g']
    X_pred_g = np.column_stack([t_grid, np.full_like(t_grid, g_wave)])
    y_pred_g = gp.predict(X_pred_g) * y_scale

    peak_idx = np.argmax(y_pred_g)
    peak_time = t_grid[peak_idx]
    peak_flux = y_pred_g[peak_idx]
    threshold = peak_flux / 2.512

    # --- 3. PERCENTILE RATIOS ---
    positive_flux = y_pred_g[y_pred_g > 0]
    
    if len(positive_flux) > 0:
        perc_20 = np.percentile(positive_flux, 20)
        perc_50 = np.percentile(positive_flux, 50)
        perc_80 = np.percentile(positive_flux, 80)
        percentile_ratio_20_50 = perc_20 / (perc_50 + 1e-9)
        percentile_ratio_80_max = perc_80 / (peak_flux + 1e-9)
    else:
        percentile_ratio_20_50 = 0.0
        percentile_ratio_80_max = 0.0

    pre_peak = y_pred_g[:peak_idx]
    t_pre = t_grid[:peak_idx]

    if len(pre_peak) > 0 and np.any(pre_peak < threshold):
        drop_idx = np.where(pre_peak < threshold)[0][-1]
        rise_time = peak_time - t_pre[drop_idx]
    else:
        rise_time = peak_time - t_min

    post_peak = y_pred_g[peak_idx:]
    t_post = t_grid[peak_idx:]

    if len(post_peak) > 0 and np.any(post_peak < threshold):
        drop_idx = np.where(post_peak < threshold)[0][0]
        fade_time = t_post[drop_idx] - peak_time
    else:
        fade_time = t_max - peak_time

    # calculate physics
    tde_error, linear_decay_slope, rise_error, fade_shape, fwhm, pre_peak_var = calculate_physics_wars(t_grid, y_pred_g, peak_idx, peak_time, peak_flux)
    match_tde = calculate_template_matching(t_grid, y_pred_g, peak_idx, peak_time, peak_flux)

    def get_val(t, band):
        val = gp.predict([[t, FILTER_WAVELENGTHS[band]]])[0] * y_scale
        return val if val > 0 else 1e-5

    val_u = get_val(peak_time, 'u')
    val_g = get_val(peak_time, 'g')
    val_r = get_val(peak_time, 'r')
    
    flux_ratio_ug = val_u / val_g 
    flux_ratio_gr = val_g / val_r 

    ug_peak = -2.5 * np.log10(val_u / val_g)
    gr_peak = -2.5 * np.log10(val_g / val_r)
    ur_peak = -2.5 * np.log10(val_u / val_r)

    # COLOR FEATURES
    t_samples = np.linspace(peak_time, peak_time + fade_time, 5)
    g_samples = [get_val(t, 'g') for t in t_samples]
    r_samples = [get_val(t, 'r') for t in t_samples]
    gr_colors = [-2.5 * np.log10(g/r) for g, r in zip(g_samples, r_samples)]
    
    mean_color_gr = np.mean(gr_colors)
    std_color_gr = np.std(gr_colors)
    
    color_monotonicity, _ = spearmanr(np.arange(5), gr_colors)
    if np.isnan(color_monotonicity):
         color_monotonicity = 0.0
    
    try:
        slope, _, _, _, _ = linregress(np.arange(5), gr_colors)
        color_slope_gr = slope
    except Exception:
        color_slope_gr = 0.0

    t_fade_pt = peak_time + (fade_time/2)
    gr_fade = -2.5 * np.log10(get_val(t_fade_pt, 'g') / get_val(t_fade_pt, 'r'))
    color_cooling_rate = gr_fade - gr_peak 
    
    def get_area(band):
        y_band = gp.predict(np.column_stack([t_grid, np.full_like(t_grid, FILTER_WAVELENGTHS[band])])) * y_scale
        return np.trapezoid(y_band, t_grid)
    
    total_area = get_area('u') + get_area('g') + get_area('r') + get_area('i')
    
    rise_fade_ratio = rise_time / fade_time if fade_time > 0 else 0
    area_under_curve = np.trapezoid(y_pred_g, t_grid)
    compactness = area_under_curve / peak_flux if peak_flux > 0 else 0
    rise_slope = amplitude / rise_time if rise_time > 1 else amplitude
    
    baseline_window = int(len(y_pred_g) * 0.15)
    baseline_flux = np.median(y_pred_g[:baseline_window])
    baseline_ratio = baseline_flux / peak_flux if peak_flux > 0 else 0

    return {
        'ObjectId': obj_id,
        'amplitude': amplitude,
        'ls_time': ls_time,
        'ls_wave': ls_wave,
        'rise_time': rise_time,
        'fade_time': fade_time,
        'fwhm': fwhm,
        'rise_fade_ratio': rise_fade_ratio,
        'compactness': compactness,
        'rise_slope': rise_slope,
        
        # Physics
        'tde_power_law_error': tde_error,
        'template_chisq_tde': match_tde,
        'linear_decay_slope': linear_decay_slope, 
        
        'mean_color_gr': mean_color_gr,
        'std_color_gr': std_color_gr,
        
        'total_radiated_energy_proxy': total_area, 
        'color_monotonicity': color_monotonicity, 
        'negative_flux_fraction': negative_flux_fraction,
        'percentile_ratio_20_50': percentile_ratio_20_50, 
        'percentile_ratio_80_max': percentile_ratio_80_max, 
        'rise_fireball_error': rise_error,
        'pre_peak_var': pre_peak_var, 
        'reduced_chi_square': reduced_chi_square,
        'fade_shape_correlation': fade_shape,
        'baseline_ratio': baseline_ratio,
        'color_cooling_rate': color_cooling_rate,
        'color_slope_gr': color_slope_gr, 
        'ug_peak': ug_peak,
        'gr_peak': gr_peak,
        'ur_peak': ur_peak,
        'flux_ratio_ug': flux_ratio_ug,
        'flux_ratio_gr': flux_ratio_gr,
        'flux_kurtosis': flux_kurtosis,
        'flux_skew': flux_skew, 
        'robust_duration': robust_duration,
        'duty_cycle': duty_cycle
    }


In [6]:
# get_gp_features(64073152, obj)

In [7]:
from joblib import Parallel, delayed


def compute_all_features(df, njobs=-1):
        
    def process_row(row):
        obj_df = pl.DataFrame({k: [v] for k, v in row.items()})
        return get_gp_features(row['ObjectId'], obj_df)
        
    
    results = Parallel(n_jobs=njobs)(
        delayed(process_row)(row) for row in df.iter_rows(named=True)
    )
    
    
    features_df = pl.DataFrame(results)

    # Determine redshift key
    z_key = None
    if 'Z' in df.columns:
        z_key = 'Z'
    elif 'HOSTGAL_PHOTOZ' in df.columns:
        z_key = 'HOSTGAL_PHOTOZ'
    
    if z_key is not None:
        print("Merging Redshift & Calculating Luminosity...")
        
        # Join redshift into features_df
        features_df = features_df.join(
            df.select(['ObjectId', z_key]).rename({z_key: 'redshift'}),
            on='ObjectId',
            how='left'
        )
        
        features_df = features_df.with_columns([
            # Clip replaces negative/NaN-like redshifts (e.g. sentinel values like -9)
            pl.col('redshift').clip(lower_bound=0.0).fill_null(0.0).alias('safe_z')
        ]).with_columns([
     
import torch.nn as nn       (pl.col('rise_time') / (1.0 + pl.col('safe_z'))).alias('rest_rise_time'),
            (pl.col('fade_time') / (1.0 + pl.col('safe_z'))).alias('rest_fade_time'),
            (pl.col('fwhm')      / (1.0 + pl.col('safe_z'))).alias('rest_fwhm'),
            
            (-2.5 * (pl.col('amplitude').clip(lower_bound=0.001).log(base=10))
                   - 5.0 * (pl.col('safe_z') + 0.001).log(base=10)
            ).alias('absolute_magnitude_proxy'),
            
            (pl.col('total_radiated_energy_proxy') * (pl.col('safe_z') + 0.001) ** 2
            ).alias('total_radiated_energy'),
            
            ((pl.col('tde_power_law_error') + 1e-9).log(base=10)
            ).alias('log_tde_error'),
        ]).drop('safe_z')  # drop intermediate column if you don't want it


    df = df.join(features_df, on='ObjectId', how='left')
    return df, features_df

In [75]:
## Full code from https://github.com/maymeridian/Tidal-Disruption-Classification/blob/competition_winner/src/machine_learning/model_factory.py

'''
src/machine_learning/model_factory.py
Description: Hybrid Ensemble Classifier.
'''

import numpy as np
import json
import os
from catboost import CatBoostClassifier 
from sklearn.neural_network import MLPClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score
from sklearn.base import BaseEstimator, ClassifierMixin



# Root directory of the project
BASE_DIR = './lead1st_mallorn/'


MODEL_CONFIG = {
    'default_model': 'catboost',
    'random_seed': 67,
    'test_size': 0.2
}
MODELS_DIR = os.path.join(BASE_DIR, 'models')


# FEATURE SUBSETS

# Subset of Morphology & Temporal Evolution Features
MORPHOLOGY_FEATURES = [
    'rest_rise_time', 
    'rest_fade_time', 
    'rest_fwhm', 
    'ls_time',             
    'rise_fade_ratio', 
    'compactness', 
    'rise_slope', 
    'flux_kurtosis', 
    'robust_duration', 
    'duty_cycle', 
    'pre_peak_var', 
    'amplitude'
]

# Subset of Physics & Color Metrics
PHYSICS_FEATURES = [
    'tde_power_law_error', 
    'template_chisq_tde',
    'linear_decay_slope',
    'mean_color_gr',
    'std_color_gr',
    'total_radiated_energy',
    'color_monotonicity',
    'negative_flux_fraction',
    'rise_fireball_error', 
    'reduced_chi_square', 
    'ls_wave',             
    'fade_shape_correlation', 
    'baseline_ratio', 
    'color_cooling_rate', 
    'color_slope_gr', 
    'flux_ratio_ug', 
    'flux_ratio_gr', 
    'ug_peak', 'gr_peak', 'ur_peak', 
    'redshift', 
    'absolute_magnitude_proxy', 
    'log_tde_error'
]

class EnsembleClassifier(BaseEstimator, ClassifierMixin):
    def __init__(self, scale_pos_weight=1.0):
        self.scale_pos_weight = scale_pos_weight
        self.models = {} 
        self.feature_importances_ = None

    def fit(self, X, y):
        seed = MODEL_CONFIG['random_seed']
        
        # Gradient Boosting Parameters
        cb_params = {
            'iterations': 1000, 'depth': 5, 'learning_rate': 0.02,
            'l2_leaf_reg': 10, 'rsm': 0.5, 'loss_function': 'Logloss',
            'verbose': 0, 'allow_writing_files': False,
            'random_seed': seed, 'scale_pos_weight': self.scale_pos_weight
        }
        
        # Load optimized hyperparameters if available
        json_path = os.path.join(MODELS_DIR, 'best_params.json')
        if os.path.exists(json_path):
            try:
                with open(json_path, 'r') as f: 
                    tuned = json.load(f)
                if 'scale_pos_weight' in tuned: 
                    del tuned['scale_pos_weight']
                cb_params.update(tuned)
            except Exception: 
                pass

        # Base Model : CatBoost
        self.models['base'] = CatBoostClassifier(**cb_params)
        self.models['base'].fit(X, y)
        self.feature_importances_ = self.models['base'].feature_importances_

        # Morphology Specialist Model (CatBoost on only Shape Features)
        cols_morph = [c for c in MORPHOLOGY_FEATURES if c in X.columns]
        if cols_morph:
            self.models['morphology'] = CatBoostClassifier(**cb_params)
            self.models['morphology'].fit(X[cols_morph], y)

        # Physics Specialist (CatBoost on only Physics Features)
        cols_phys = [c for c in PHYSICS_FEATURES if c in X.columns]
        if cols_phys:
            self.models['physics'] = CatBoostClassifier(**cb_params)
            self.models['physics'].fit(X[cols_phys], y)

        # Supporting Model A: Multi-Layer Perceptron (Neural Network)
        self.models['mlp'] = Pipeline([
            ('var_filter', VarianceThreshold(threshold=0.0)),  # drop within-fold constants
            ('imputer', SimpleImputer(strategy='mean')),
            ('scaler', StandardScaler()),
            ('net', MLPClassifier(hidden_layer_sizes=(64, 32), activation='relu', 
                                  solver='adam', alpha=0.01, max_iter=600, 
                                  random_state=seed))
        ])
        self.models['mlp'].fit(X, y)

        # Supporting Model B: K-Nearest Neighbors
        self.models['knn'] = Pipeline([
            ('var_filter', VarianceThreshold(threshold=0.0)),
            ('imputer', SimpleImputer(strategy='mean')),
            ('scaler', StandardScaler()),
            ('knn', KNeighborsClassifier(n_neighbors=15, weights='distance', p=2))
        ])
        self.models['knn'].fit(X, y)
        
        return self

    def predict_proba(self, X):
        # Generate Component Predictions
        p_base = self.models['base'].predict_proba(X)[:, 1]
        
        p_morph = p_base # Fallback
        if 'morphology' in self.models:
            cols = [c for c in MORPHOLOGY_FEATURES if c in X.columns]
            p_morph = self.models['morphology'].predict_proba(X[cols])[:, 1]
            
        p_phys = p_base # Fallback
        if 'physics' in self.models:
            cols = [c for c in PHYSICS_FEATURES if c in X.columns]
            p_phys = self.models['physics'].predict_proba(X[cols])[:, 1]
            
        p_mlp = self.models['mlp'].predict_proba(X)[:, 1]
        p_knn = self.models['knn'].predict_proba(X)[:, 1]
        
        # Ensemble Aggregation (Weighted Average)
        # Weights logic: 
        # - Gradient Boosting (80% Total): Split 48% Base, 16% Morph, 16% Phys
        # - Support Models (20% Total): Split 10% MLP, 10% KNN
        
        final_prob = (0.48 * p_base) + \
                     (0.16 * p_morph) + \
                     (0.16 * p_phys) + \
                     (0.10 * p_mlp) + \
                     (0.10 * p_knn)
        
        return np.vstack([1 - final_prob, final_prob]).T

    def predict(self, X):
        probs = self.predict_proba(X)[:, 1]
        return (probs >= 0.5).astype(int)

# FACTORY
def get_model(model_name, scale_pos_weight=1.0):
    if model_name == 'catboost':
        return EnsembleClassifier(scale_pos_weight=scale_pos_weight)
    elif model_name =='linearl1':
        return sklearn.svm.LinearSVC('l1', class_weight=scale_pos_weight)
    elif model_name =='linearl2':
        return sklearn.svm.LinearSVC('l2', class_weight=scale_pos_weight)
    else:
        raise ValueError("Only 'catboost' is supported.")

        

def train_with_cv(model_name, X, y):
    print("\n--- Initializing Hybrid Ensemble Classifier ---")
    print("    Configuration: Gradient Boosting (Core) + MLP/KNN (Support)")
    
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=MODEL_CONFIG['random_seed'])
    cv_scores = []
    best_thresholds = []

    fold = 1
    for train_index, val_index in skf.split(X, y):
        X_train, X_val = X.iloc[train_index], X.iloc[val_index]
        y_train, y_val = y.iloc[train_index], y.iloc[val_index]
        
        n_pos = y_train.sum()
        scale_weight = (len(y_train) - n_pos) / n_pos if n_pos > 0 else 1.0

        model = get_model(model_name, scale_pos_weight=scale_weight)
        model.fit(X_train, y_train)
        
        probs_val = model.predict_proba(X_val)[:, 1]
        
        best_f1 = 0.0
        best_t = 0.5
        for t in np.arange(0.2, 0.8, 0.02):
            s = f1_score(y_val, (probs_val >= t).astype(int), zero_division=0) 
            if s > best_f1: 
                best_f1 = s
                best_t = t
        
        val_tdes = y_val.sum()
        print(f"   Fold {fold}: F1={best_f1:.4f} (Thresh={best_t:.2f}) - Validation Positives: {val_tdes}")
        
        cv_scores.append(best_f1)
        best_thresholds.append(best_t)
        fold += 1

    avg_f1 = np.mean(cv_scores)
    avg_thresh = np.mean(best_thresholds)
    
    print(f"\n   Average Ensemble F1: {avg_f1:.4f}")
    
    # Final Train
    print("\n--- Training Final Production Model ---")
    n_pos_all = y.sum()
    final_weight = (len(y) - n_pos_all) / n_pos_all
    
    final_model = get_model(model_name, scale_pos_weight=final_weight)
    final_model.fit(X, y)
    
    return final_model, avg_f1, avg_thresh

In [9]:
def deextin_quality_features_clean(parq, njobs=-1, apply_quality=True):
    # This might need adjustement if binary_class is not deifned
    
    ela_deext = apply_deextinction_parq(parq)
    
    print(len(ela_deext))

    if apply_quality:
        ela_deext = apply_quality_cuts(ela_deext)
    print('--"quality cut"--')
    
    print(len(ela_deext))
    
    t = time.time()
    parq_df_features, features_only = compute_all_features(ela_deext, njobs=njobs)
    
    print(f" Computing featurestook {time.time()-t}")
    features_pd = parq_df_features[['ObjectId', 'binary_class']].join(features_only, on='ObjectId', how='left').to_pandas()
    
    X = features_pd.drop(columns=['ObjectId', 'binary_class'])
    y = features_pd['binary_class']
    
    # Clip extreme values before passing to model... should we clip before or after Var? maybe before?
    X = X.clip(lower=-1e10, upper=1e10)
    
    # This should actually not be done on test blindly... but in practice we always remove linear_slope_decay stuff so i guess whatev   
    selector = VarianceThreshold(threshold=0.0)
    X_arr = selector.fit_transform(X)
    kept_columns = X.columns[selector.get_support()]
    dropped_columns = X.columns[~selector.get_support()]
    print("Dropped constant columns:", dropped_columns.tolist())
    X = pd.DataFrame(X, columns=kept_columns)
    
    X = X.replace([np.inf, -np.inf], np.nan)

    return X, y, ela_deext

## Get CLS function

In [83]:
    
def get_CLS_traintest(modeldir, parqname):
    
    env_file = modeldir.split('_checkpoint')[0]+'_config.env'
    
    config = MallornConfigContrastiveEnv(
        _env_file=env_file
    )
    encoder_args = override_encoder_size(config.model_size)
    encoder_args["drop_path_rate"]=0.15
    encoder_args["hidden_drop_rate"]=0.1
    encoder_args["pos_drop_rate"]=0.1
    encoder_args["attn_drop_rate"]=0.1
    encoder_args["attn_proj_drop_rate"]=0.1
    
    decoder_size = config.decoder_size or encoder_args['d_model']
    decoder_args = {
        "d_model": decoder_size,
        "nhead": 3,
        "depth": 2,
        "drop_path_rate": 0.05
    }

    model_config = RoMAEForPreTrainingConfig(
        encoder_config=EncoderConfig(**encoder_args),
        decoder_config=EncoderConfig(**decoder_args),
        tubelet_size=(1, 1, 1),
        n_channels=2,
        n_pos_dims=2,
    )

    model = RoMAEPreTrainingContrastive(
        config=model_config,
        contrastive_config=config,
        augmentation_fn=None, # shouldnt be used
    )

    
    model = load_from_checkpoint_contrastive(modeldir,
                            RoMAEPreTrainingContrastive, RoMAEForPreTrainingConfig, config)
    
    ## probably need to have this in config
    model.set_loss_fn(nn.HuberLoss(reduction='none', delta=0.5)) # delta: need to check the actual flux distrib now that we're rescaling?

    dataset = MallornDatasetwLabelTrimMask(parqname,
                                         mask_ratio=0.0, 
                                         gaussian_noise = False,
                                         obs_dropout_end_trim = 0.0,      # fraction of seq to trim from ends
                                         obs_dropout_edge_erosion = 0.0,  # max fraction to erode per gap edge
                                         gap_threshold_factor = 0.0,       # in MJD?
                                         random_dropout_ratio = 0.0,
                                         training = False)
    
    _dataloader = torch.utils.data.DataLoader(
            dataset,
            batch_size=256,
            num_workers=1,
            pin_memory=True,
            collate_fn=torch.utils.data.dataloader.default_collate,
            prefetch_factor=2,
            shuffle=False
        )

    model.eval()

    cls_full = []
    cls_contr = []
    labels = []
    ## bjectid?

    for datal in _dataloader:
        with torch.no_grad():
            out = model.forward_cls(datal['values'], datal['mask'],
                    datal['positions'], datal['pad_mask'],
                    decode=False)
            cls_full.append(out['embeddings'][:,0,:].cpu().numpy())
            cls_contr.append(out['embeddings'][:,0,:].cpu().numpy()[:,0:config.cls_contrastive_dim].reshape(-1,config.cls_contrastive_dim))
            labels.append(datal['label'].cpu().numpy())
            torch.cuda.empty_cache()
            
    cls_full  = np.concatenate(cls_full,  axis=0)  # [dataset_size, cls_size]
    cls_contr   = np.concatenate(cls_contr,   axis=0)  # [dataset_size, 32]
    labels    = np.concatenate(labels,    axis=0)  # [dataset_size]


    
    objids = pl.read_parquet(parqname)['ObjectId'].to_numpy()

    
    torch.cuda.empty_cache()
    del model, dataset, _dataloader
    
    return cls_full, cls_contr, labels,  objids

In [77]:
import numpy as np
from sklearn.svm import LinearSVC
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    classification_report, f1_score, roc_auc_score, make_scorer
)
from sklearn.pipeline import Pipeline
import pandas as pd
import warnings

warnings.filterwarnings("ignore")


def kfold_linearsvc_search(
    X, y,
    C_values=(0.0001, 0.001, 0.01, 0.1, 1.0, 10.0),
    penalties=("l1", "l2"),
    n_splits=5,
    class_weight="balanced",   # handles class imbalance in training
    scoring="f1",              # better than accuracy for imbalanced data
    random_state=42,
    max_iter=5000,
    verbose=True,
):
    """
    Grid-search LinearSVC over penalty type (l1/l2) and C values using
    stratified k-fold cross-validation.
    `class_weight='balanced'` makes the
    SVM up-weight minority-class errors during training.

    Parameters
    ----------
    X            : array-like, shape (n_samples, n_features)
    y            : array-like, shape (n_samples,)
    C_values     : iterable of floats  – regularisation strengths to try
                   (smaller C = stronger regularisation = less overfitting)
    penalties    : iterable of str    – 'l1' (sparse) and/or 'l2' (default)
    n_splits     : int                – number of CV folds
    class_weight : 'balanced' or dict – passed straight to LinearSVC
    scoring      : str                – 'f1', 'f1_macro', 'roc_auc', etc.
    random_state : int
    max_iter     : int                – increase if LinearSVC warns about convergence
    verbose      : bool               – print a results table

    Returns
    -------
    results_df   : pd.DataFrame  – per-config mean ± std of the scoring metric
    best_config  : dict          – {'penalty', 'C', 'mean_score', 'std_score'}
    best_model   : Pipeline      – scaler + LinearSVC retrained on ALL data
    """
    cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)

    rows = []

    for penalty in penalties:
        # l1 requires the 'liblinear' solver; l2 works with both
        dual = (penalty == "l2")

        for C in C_values:
            pipe = Pipeline([
                ("scaler", StandardScaler()),          # always scale for SVMs
                ("svc", LinearSVC(
                    penalty=penalty,
                    C=C,
                    dual=dual,
                    class_weight=class_weight,
                    max_iter=max_iter,
                    random_state=random_state,
                )),
            ])

            fold_scores = []
            for train_idx, val_idx in cv.split(X, y):
                X_tr, X_val = X[train_idx], X[val_idx]
                y_tr, y_val = y[train_idx], y[val_idx]
                ## To make consistant with BestMall code that takes pd DF / series
                # X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
                # y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]

                pipe.fit(X_tr, y_tr)
                y_pred = pipe.predict(X_val)

                if scoring == "roc_auc":
                    # LinearSVC has decision_function, not predict_proba
                    score = roc_auc_score(y_val, pipe.decision_function(X_val))
                elif scoring == "f1_macro":
                    score = f1_score(y_val, y_pred, average="macro", zero_division=0)
                else:  # default: binary f1
                    score = f1_score(y_val, y_pred, average="binary", zero_division=0)

                fold_scores.append(score)

            rows.append({
                "penalty": penalty,
                "C": C,
                "mean_score": np.mean(fold_scores),
                "std_score":  np.std(fold_scores),
            })

    results_df = pd.DataFrame(rows).sort_values("mean_score", ascending=False)

    if verbose:
        print(f"\n{'='*55}")
        print(f"  LinearSVC grid search  |  metric: {scoring}  |  folds: {n_splits}")
        print(f"{'='*55}")
        print(results_df.to_string(index=False, float_format="{:.4f}".format))
        print(f"{'='*55}\n")

    best = results_df.iloc[0]
    best_config = {
        "penalty":    best["penalty"],
        "C":          best["C"],
        "mean_score": best["mean_score"],
        "std_score":  best["std_score"],
    }

    # Retrain the winning config on the full dataset
    best_model = Pipeline([
        ("scaler", StandardScaler()),
        ("svc", LinearSVC(
            penalty=best_config["penalty"],
            C=best_config["C"],
            dual=(best_config["penalty"] == "l2"),
            class_weight=class_weight,
            max_iter=max_iter,
            random_state=random_state,
        )),
    ])
    best_model.fit(X, y)

    if verbose:
        print(f"Best config  →  penalty={best_config['penalty']},  "
              f"C={best_config['C']},  "
              f"{scoring}={best_config['mean_score']:.4f} ± {best_config['std_score']:.4f}")
        print("\nFull-data classification report (best model on train set):")
        print(classification_report(y, best_model.predict(X), zero_division=0))

    return results_df, best_config, best_model

In [11]:

parq_all300 = pl.read_parquet('/ceph/hpc/home/contardog/ELASTICC2/mallorn-like/training_ELAsTiCC2_sub_300pos_3000neg.parq')

#parq_all50  = pl.read_parquet('/ceph/hpc/home/contardog/ELASTICC2/mallorn-like/training_ELAsTiCC2_sub_50pos_3000neg.parq')

In [12]:
#np.sum(np.isin(parq_all50['ObjectId'].to_numpy(), parq_all300['ObjectId']))

In [13]:
parq_all300 = parq_all300.filter(pl.col('binary_class')!=-1)

## If features have not been computed, uncomment below

In [14]:
# features_BestMall_all300, labels_all300, parq_all300_deext = deextin_quality_features_clean(parq_all300)
# ## Save to avoid redoing each time
# features_BestMall_all300.to_csv('/ceph/hpc/home/contardog/ELASTICC2/mallorn-like/training_ELAsTiCC2_sub_300pos_3000neg_BestMallfeat.csv')
# labels_all300.to_csv('/ceph/hpc/home/contardog/ELASTICC2/mallorn-like/training_ELAsTiCC2_sub_300pos_3000neg_BestMallfeat_lab.csv')
# parq_all300_deext.write_parquet('/ceph/hpc/home/contardog/ELASTICC2/mallorn-like/training_ELAsTiCC2_sub_300pos_3000neg_BestMallfeat.parq')

In [15]:
# parq_test = pl.read_parquet('/ceph/hpc/home/contardog/ELASTICC2/mallorn-like/test_ELAsTiCC_10000ex_5.0percentTDE.parquet')

# ## I removed the quality cut although we could keep it but need to keep track of who gets removed and decide to flag it as not TDE
# features_BestMall_test, labels_test_bestmall, parq_test_deext = deextin_quality_features_clean(parq_test, njobs=1, apply_quality=False)
# ## Save to avoid redoing each time
# features_BestMall_test.to_csv('/ceph/hpc/home/contardog/ELASTICC2/mallorn-like/test_ELAsTiCC_10000ex_5per _BestMallfeat.csv')
# labels_test_bestmall.to_csv('/ceph/hpc/home/contardog/ELASTICC2/mallorn-like/test_ELAsTiCC_10000ex_5per_BestMallfeat_lab.csv')
# parq_test_deext.write_parquet('/ceph/hpc/home/contardog/ELASTICC2/mallorn-like/test_ELAsTiCC_10000ex_5per_BestMallfeat.parq')

## If features have been precomputed:

In [16]:
features_BestMall_all300 = pd.read_csv('/ceph/hpc/home/contardog/ELASTICC2/mallorn-like/training_ELAsTiCC2_sub_300pos_3000neg_BestMallfeat.csv')
labels_all300 = pd.read_csv('/ceph/hpc/home/contardog/ELASTICC2/mallorn-like/training_ELAsTiCC2_sub_300pos_3000neg_BestMallfeat_lab.csv')['binary_class']
parq_all300_deext = pl.read_parquet('/ceph/hpc/home/contardog/ELASTICC2/mallorn-like/training_ELAsTiCC2_sub_300pos_3000neg_BestMallfeat.parq')

In [17]:

features_BestMall_test = pd.read_csv('/ceph/hpc/home/contardog/ELASTICC2/mallorn-like/test_ELAsTiCC_10000ex_5per _BestMallfeat.csv')
labels_test_bestmall = pd.read_csv('/ceph/hpc/home/contardog/ELASTICC2/mallorn-like/test_ELAsTiCC_10000ex_5per_BestMallfeat_lab.csv')['binary_class']
parq_test_deext = pl.read_parquet('/ceph/hpc/home/contardog/ELASTICC2/mallorn-like/test_ELAsTiCC_10000ex_5per_BestMallfeat.parq')

In [18]:
# model_bestmall, score_bestmal, threshold_bestmal = train_with_cv('catboost', features_BestMall_all300, labels_all300['binary_class'])

In [19]:
# predte = model_bestmall.predict(features_BestMall_test)
# print(classification_report(labels_test_bestmall, predte))
# print('----------------------')
# print(classification_report(labels_test_bestmall, predte>threshold_bestmal))

In [20]:
# supconrun_selfsuponly  supconrun_selfsupsupcon_1-1 supconrun_selfsupsupcon_1-05  supconrun_supcononly
# _tinyermidshallow6_1weight_[temp007]_[nounsup]_checkpoint_val

In [ ]:

### Loop over datasets and models to compute F1 etc (keep models?) 
### Maybe load 300 pos and subsample to save time / or find objectID from the sample?
parq_testname = '/ceph/hpc/home/contardog/ELASTICC2/mallorn-like/test_ELAsTiCC_10000ex_5.0percentTDE.parquet'
mainfold = '/ceph/hpc/home/contardog/MALLORN/SupCon_logs/ELASTICC2_mallornlike'

models_classif= {}
preds_wiwo_thresh = {}
f1scores = {}
classifreports = {}

linears = {'models':{}, 'f1scores':{}, 'classifreports':{}, 'preds':{}}


for kpos in [50,100,150,200,250,300]:

    print("==========********============="*3)
    print(f"POSITIVE EXAMPLES {kpos}")
    print("==========********============="*3)
    #parq = pl.read_parquet('/ceph/hpc/home/contardog/ELASTICC2/mallorn-like/training_ELAsTiCC2_subtrain_100pos_3000neg_nounsup.parq')
    
    parq_kname = f'/ceph/hpc/home/contardog/ELASTICC2/mallorn-like/training_ELAsTiCC2_sub_{kpos}pos_3000neg.parq'
    parq_k = pl.read_parquet(parq_kname)
    parq_k = parq_k.filter(pl.col('binary_class')!=-1)
    
    ## Get the subset for training, features are already precomputed ; this 
    mask_kpo = np.isin(parq_all300_deext['ObjectId'].to_numpy(), parq_k['ObjectId'])
    features_kpo = features_BestMall_all300[mask_kpo] # checkk
    labels_kpo = labels_all300[mask_kpo]

    
    ## Get best model with BestMall features
    model_bestmall, score_bestmal, threshold_bestmal = train_with_cv('catboost', features_kpo, labels_kpo)
    pred_test_kpo_bestmal = model_bestmall.predict(features_BestMall_test)
    pred_test_kpo_bestmal_thresh = pred_test_kpo_bestmal > threshold_bestmal
    
    f1_ = f1_score(labels_test_bestmall, pred_test_kpo_bestmal_thresh)
    clasreport_ = classification_report(labels_test_bestmall, pred_test_kpo_bestmal_thresh)
    print(" BEST MALLORN PERF: ")
    print(clasreport_)
    
    if 'bestmall' not in models_classif:
        models_classif['bestmall'] = []
        preds_wiwo_thresh['bestmall'] = []
        f1scores['bestmall'] = []
        classifreports['bestmall'] = []
        
    models_classif['bestmall'].append([model_bestmall, score_bestmal, threshold_bestmal])
    preds_wiwo_thresh['bestmall'].append([pred_test_kpo_bestmal, pred_test_kpo_bestmal_thresh])
    f1scores['bestmall'].append(f1_)
    classifreports['bestmall'].append(clasreport_)

    ## for us: is it shacky to merge train/val in this Kfold?
    ## Try it for now, otherwise we have to implem our own validation process which is a bit annoying.
    print("====="*5)
    
    ## For all models, trained with this dataset: (make this more elegant/prettier)
    for fold in ['supconrun_selfsupsupcon_1-1', 'supconrun_selfsupsupcon_1-05', 'supconrun_supcononly','supconrun_selfsuponly']:
        for adon in ['', '_temp007', '_temp007_nounsup']:
            fold_best = f"{mainfold}/{fold}/_elasticc2_{kpos}pos_3000neg_tinyermidshallow6_1weight{adon}_checkpoint_val/_bestval"

            # 
            # compute the cls token for train and test
            # if os.path.isfile(f"{mainfold}/{fold}/_elasticc2_{kpos}pos_3000neg_tinyermidshallow6_1weight{adon}_clsfull.npy"):
            #     cls_full = np.load(f"{mainfold}/{fold}/_elasticc2_{kpos}pos_3000neg_tinyermidshallow6_1weight{adon}_clsfull.npy")
            #     cls_contr = 
                
            cls_full, cls_contr, labels_cls,  objids = get_CLS_traintest(fold_best, parq_kname)
            ## need to remove the unsup: 
            mask_ = labels_cls!=-1
            cls_full = cls_full[mask_]
            cls_contr = cls_contr[mask_]
            labels_cls = labels_cls[mask_]
            objids = objids[mask_]
            
            cls_fulltest, cls_contrtest, labels_cls_test, objids_test = get_CLS_traintest(fold_best, parq_testname)

            ## save cls to save time later?
            np.save(f"{mainfold}/{fold}/_elasticc2_{kpos}pos_3000neg_tinyermidshallow6_1weight{adon}_clsfull.npy", cls_full)
            np.save(f"{mainfold}/{fold}/_elasticc2_{kpos}pos_3000neg_tinyermidshallow6_1weight{adon}_clsfull_test.npy", cls_fulltest)

            
            if fold == 'supconrun_selfsuponly':
                results_df, best_config, model_CLS = kfold_linearsvc_search(cls_full, labels_cls)
                pred_test_kpo_CLS = model_CLS.predict(cls_fulltest)
            else:
                results_df, best_config, model_CLS = kfold_linearsvc_search( cls_contr, labels_cls)
                pred_test_kpo_CLS = model_CLS.predict(cls_contrtest)

            
            f1_ = f1_score(labels_cls_test, pred_test_kpo_CLS)
            clasreport_ = classification_report(labels_cls_test, pred_test_kpo_CLS)

            print(f"{fold}_{adon} PERF WITH LINEAR")
            print(clasreport_)
            print("---"*5)

            #linears = {'models':{}, 'f1scores':{}, 'classifreports':{}, 'preds':{}}

            if f"{fold}_{adon}" not in linears['models']:
                linears['models'][f"{fold}_{adon}"] = []
                linears['f1scores'][f"{fold}_{adon}"] = []
                linears['classifreports'][f"{fold}_{adon}"] = []
                linears['preds'][f"{fold}_{adon}"] = []
                
            linears['models'][f"{fold}_{adon}"].append([model_CLS, results_df, best_config])
            linears['preds'][f"{fold}_{adon}"].append([pred_test_kpo_CLS])
            linears['f1scores'][f"{fold}_{adon}"].append(f1_)
            linears['classifreports'][f"{fold}_{adon}"].append(clasreport_)




            ### Trying with the CatBoost from BestMall pipeline
            cls_full = pd.DataFrame(cls_full)
            cls_fulltest = pd.DataFrame(cls_fulltest)
            cls_contr = pd.DataFrame(cls_contr)
            cls_contrtest = pd.DataFrame(cls_contrtest)
            labels_cls = pd.Series(labels_cls)

            if fold == 'supconrun_selfsuponly':
                model_CLS, score_CLS, threshold_CLS = train_with_cv('catboost', cls_full, labels_cls)
                pred_test_kpo_CLS = model_CLS.predict(cls_fulltest)
                pred_test_kpo_CLS_trhesh = pred_test_kpo_CLS > threshold_CLS
            else:
                model_CLS, score_CLS, threshold_CLS = train_with_cv('catboost', cls_contr, labels_cls)
                pred_test_kpo_CLS = model_CLS.predict(cls_contrtest)
                pred_test_kpo_CLS_trhesh = pred_test_kpo_CLS > threshold_CLS

            
            f1_ = f1_score(labels_cls_test, pred_test_kpo_CLS_trhesh)
            clasreport_ = classification_report(labels_cls_test, pred_test_kpo_CLS_trhesh)

            print(f"{fold}_{adon} PERF")
            print(clasreport_)
            print("====="*5)
            
            if f"{fold}_{adon}" not in models_classif:
                models_classif[f"{fold}_{adon}"] = []
                preds_wiwo_thresh[f"{fold}_{adon}"] = []
                f1scores[f"{fold}_{adon}"] = []
                classifreports[f"{fold}_{adon}"] = []
                
            models_classif[f"{fold}_{adon}"].append([model_CLS, score_CLS, threshold_CLS])
            preds_wiwo_thresh[f"{fold}_{adon}"].append([pred_test_kpo_CLS, pred_test_kpo_CLS_trhesh])
            f1scores[f"{fold}_{adon}"].append(f1_)
            classifreports[f"{fold}_{adon}"].append(clasreport_)
            

    
    ## Update BestMallorn code also to have a linear (?) SVM or something; 

    ## Error bars: reiter with different seeds; different training sets? 

    ## 'small number statistics': compare F1 on 10k/500 vs 2k/100 subsampled 
    
    
    
    
    
    
    
    

==========********=======================********=======================********=============
POSITIVE EXAMPLES 50
==========********=======================********=======================********=============

--- Initializing Hybrid Ensemble Classifier ---
    Configuration: Gradient Boosting (Core) + MLP/KNN (Support)
   Fold 1: F1=0.8571 (Thresh=0.30) - Validation Positives: 10
   Fold 2: F1=0.4615 (Thresh=0.22) - Validation Positives: 10
   Fold 3: F1=0.9000 (Thresh=0.30) - Validation Positives: 10
   Fold 4: F1=0.7000 (Thresh=0.24) - Validation Positives: 10
   Fold 5: F1=0.7368 (Thresh=0.62) - Validation Positives: 10

   Average Ensemble F1: 0.7311

--- Training Final Production Model ---
 BEST MALLORN PERF: 
              precision    recall  f1-score   support

           0       0.98      1.00      0.99      9500
           1       0.93      0.71      0.80       500

    accuracy                           0.98     10000
   macro avg       0.96      0.85      0.90     10000
we

In [82]:
cls_fulltest

,0,1,2,3,4,5,6,7,8,9,...,110,111,112,113,114,115,116,117,118,119
0,-0.726100,-0.159648,-1.031108,-1.125932,0.050565,0.281467,1.406409,1.215955,1.129198,1.371286,...,-0.043851,-0.197121,0.303513,1.559403,-0.259773,1.176455,0.639106,1.802017,-0.571826,-0.559182
1,-0.729334,0.056938,-0.981197,-0.949315,-0.039024,-0.459297,2.082802,0.805190,1.244361,0.980180,...,-1.206872,-0.138059,0.634661,1.442197,0.290437,2.222622,0.518250,2.240429,-0.403996,-1.000440
2,-0.351560,0.422782,0.012288,-0.310713,-0.107376,0.146070,1.918705,0.565471,0.583710,1.922948,...,-0.989667,0.353759,0.758994,0.878383,0.550860,1.749485,0.573085,3.761521,-0.042284,-0.798489
3,-0.457408,-0.134074,-1.053292,-1.107836,0.302185,0.604053,1.086712,1.034618,1.049927,1.762004,...,0.636014,-0.435896,0.058880,1.440150,-0.454813,0.339241,0.441591,1.222484,-0.764110,-0.052519
4,-0.442306,-0.098299,-0.713352,-0.665862,0.067396,0.467529,1.563360,0.976744,0.381943,1.723653,...,-0.285479,0.303954,0.512028,1.358286,-0.319952,1.387259,0.625911,2.371366,-0.462240,-0.597994
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,-0.390460,-0.354340,-1.156741,-0.927240,0.176599,0.682452,1.166169,1.238698,0.447881,1.702094,...,0.360694,-0.337871,0.261550,1.505764,-0.457394,0.739439,0.705058,1.408263,-0.602177,-0.445909
9996,-0.503677,-0.033665,-0.730371,-0.744403,0.041869,0.291791,1.358342,1.163573,0.511890,1.577031,...,-0.365621,0.147914,0.488415,1.232870,-0.210015,1.507654,0.608214,2.740801,-0.232087,-0.796939
9997,-0.437412,-0.115979,-0.546593,-0.649641,-0.014679,0.454337,1.530761,0.952094,0.513475,1.793762,...,-0.301801,0.087083,0.574208,1.287055,0.016152,1.525475,0.501330,2.501130,-0.613103,-0.683692
9998,-0.742066,-0.273964,-0.819182,-0.772098,0.163588,0.019761,1.214820,0.778093,1.993227,0.939618,...,-0.070542,0.128031,0.119927,0.648201,0.317733,0.450237,0.473438,1.650094,-0.695414,0.339172
